In [1]:
BASE_CONFIG = {
    "vocab_size": 50257,       # Vocabulary size
    "context_length": 1024,    # Context length
    "drop_rate": 0.1,          # Dropout rate
    "qkv_bias": True          # Query-Key-Value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

In [2]:
CHOOSE_MODEL = "gpt2-medium (355M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

In [3]:
CLASSES = ["positive", "neutral", "negative"]

In [4]:
import torch
import math
import torch.nn as nn
import tiktoken
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

In [1]:
model_file_path = "models/model_3.pth"

## From Chapter 2

In [6]:
# Dataset class used to prepare training samples for GPT-style language modeling
# Each sample consists of:
#   input sequence  -> tokens
#   target sequence -> same tokens shifted by 1 position (next-token prediction)

class GPTDatasetV1(Dataset):

    def __init__(self, txt, tokenizer, max_length, stride):
        # Lists that will store the input and target sequences
        self.input_ids = []
        self.target_ids = []

        # Convert the entire text into token IDs using the tokenizer
        token_ids = tokenizer.encode(txt)

        # Slide a window over the tokenized text to create many training samples
        # stride controls how much the window moves each step (overlap between samples)
        for i in range(0, len(token_ids) - max_length, stride):

            # Input sequence of length max_length
            input_chunk = token_ids[i:i + max_length]

            # Target sequence is the same sequence shifted by one token
            # (model learns to predict the next token)
            target_chunk = token_ids[i + 1:i + max_length + 1]

            # Store tensors for PyTorch training
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    # Returns number of training samples in the dataset
    def __len__(self):
        return len(self.input_ids)

    # Returns a single training sample (input sequence, target sequence)
    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


# Helper function that builds a PyTorch DataLoader for the dataset
def create_dataloader_v1(
    txt,
    batch_size=4,
    max_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0
):

    # Load GPT-2 tokenizer from tiktoken
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset object
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Wrap dataset in a DataLoader for batching and iteration during training
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,  # number of samples per batch
        shuffle=shuffle,        # shuffle dataset each epoch
        drop_last=drop_last,    # drop last batch if smaller than batch_size
        num_workers=num_workers # number of parallel workers for loading data
    )

    return dataloader

# Create dataloaders for training and validation datasets
def create_train_validator(file_path, config, train_ratio):

    # Read the entire dataset from the file
    with open(file_path, "r", encoding="utf-8") as file:
        text_data = file.read()
    
    # Compute dataset size statistics
    total_characters = len(text_data)              # Number of characters in dataset
    total_tokens = len(tokenizer.encode(text_data)) # Number of tokens after tokenization
    
    # Determine split index based on the train/validation ratio
    split_idx = int(train_ratio * total_characters)
    
    # Split dataset into training and validation portions
    train_data = text_data[:split_idx]
    val_data = text_data[split_idx:]
    
    # Create dataloader for the training set
    train_loader = create_dataloader_v1(
        train_data,
        batch_size=2,                              # Number of sequences per batch
        max_length=config["context_length"],       # Maximum sequence length (model context window)
        stride=config["context_length"],           # Step size when creating overlapping sequences
        drop_last=True,                            # Drop incomplete last batch for consistent training
        shuffle=True,                              # Shuffle training samples
        num_workers=0                              # Number of parallel workers for loading data
    )

    # Create dataloader for the validation set
    val_loader = create_dataloader_v1(
        val_data,
        batch_size=2,
        max_length=config["context_length"],
        stride=config["context_length"],
        drop_last=False,                           # Keep last batch for full validation coverage
        shuffle=False,                             # No shuffling for evaluation
        num_workers=0
    )

    # Return both dataloaders
    return [train_loader, val_loader]

## From Chapter 3

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        # Ensure that the output dimension can be evenly split across heads
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        # Total output dimension of attention layer
        self.d_out = d_out

        # Number of attention heads
        self.num_heads = num_heads

        # Dimension handled by each head
        self.head_dim = d_out // num_heads

        # Linear projections to produce Query, Key, and Value vectors
        # Each token embedding is projected into d_out dimension
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Final linear layer to mix the outputs of all heads
        self.out_proj = nn.Linear(d_out, d_out)

        # Dropout applied to attention weights (regularization)
        self.dropout = nn.Dropout(dropout)

        # Causal mask (upper triangular matrix)
        # Prevents tokens from attending to future tokens in autoregressive models
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):

        # x shape: (batch_size, num_tokens, input_dimension)
        b, num_tokens, d_in = x.shape

        # Project input embeddings into query, key, and value vectors
        # Shape after projection: (batch_size, num_tokens, d_out)
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Split each projection into multiple heads
        # New shape: (batch_size, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Move the head dimension before the token dimension
        # New shape: (batch_size, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        values = values.transpose(1, 2)
        queries = queries.transpose(1, 2)

        # Compute attention scores using scaled dot-product attention
        # scores shape: (batch_size, num_heads, num_tokens, num_tokens)
        attn_scores = queries @ keys.transpose(2, 3)

        # Apply causal mask so tokens cannot attend to future tokens
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        # Scale scores by sqrt(head_dim) for numerical stability
        # Then convert scores to probabilities with softmax
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5,
            dim=-1
        )

        # Apply dropout to attention weights
        attn_weights = self.dropout(attn_weights)

        # Compute weighted sum of value vectors
        # Result shape: (batch_size, num_heads, num_tokens, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # Merge all heads back together
        # Shape becomes: (batch_size, num_tokens, d_out)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)

        # Final linear projection after concatenating heads
        context_vec = self.out_proj(context_vec)

        return context_vec

## From Chapter 4

In [8]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()

        # Small constant added to variance for numerical stability
        self.eps = 1e-5

        # Learnable scaling parameter (gamma in LayerNorm literature)
        # One value per embedding dimension
        self.scale = nn.Parameter(torch.ones(emb_dim))

        # Learnable shift parameter (beta)
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        # Compute mean across embedding dimension
        # Shape: (batch, seq_len, 1)
        mean = x.mean(dim=-1, keepdim=True)

        # Compute variance across embedding dimension
        # unbiased=False matches typical LayerNorm implementation
        var = x.var(dim=-1, keepdim=True, unbiased=False)

        # Normalize input: (x - mean) / sqrt(var + eps)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)

        # Apply learnable scale and shift
        return self.scale * norm_x + self.shift


class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        # Gaussian Error Linear Unit activation
        # Smooth alternative to ReLU used in GPT/BERT
        # This is the tanh approximation of GELU
        return 0.5 * x * (
            1 + torch.tanh(
                torch.sqrt(torch.tensor(2.0 / torch.pi))
                * (x + 0.044715 * torch.pow(x, 3))
            )
        )


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # Position-wise feed-forward network
        # Expands embedding dimension then projects back
        self.layers = nn.Sequential(

            # First linear layer expands dimension (typically 4x)
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),

            # Non-linear activation
            GELU(),

            # Project back to original embedding size
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        # Applies feed-forward network to each token independently
        return self.layers(x)


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # Multi-head self-attention layer
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )

        # Position-wise feed-forward network
        self.ff = FeedForward(cfg)

        # Layer normalization before attention
        self.norm1 = LayerNorm(cfg["emb_dim"])

        # Layer normalization before feed-forward
        self.norm2 = LayerNorm(cfg["emb_dim"])

        # Dropout applied to residual connections
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):

        # ---- Self-Attention Block ----

        # Save residual (shortcut) connection
        shortcut = x

        # Apply layer normalization
        x = self.norm1(x)

        # Apply multi-head self-attention
        x = self.att(x)

        # Apply dropout
        x = self.drop_shortcut(x)

        # Add residual connection
        x = x + shortcut


        # ---- Feed-Forward Block ----

        # Save residual connection again
        shortcut = x

        # Apply second normalization
        x = self.norm2(x)

        # Apply feed-forward network
        x = self.ff(x)

        # Apply dropout
        x = self.drop_shortcut(x)

        # Add residual connection
        x = x + shortcut

        return x


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # Token embedding layer
        # Converts token IDs into embedding vectors
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])

        # Positional embedding layer
        # Adds information about token position in the sequence
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])

        # Dropout applied to embeddings
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        # Stack of Transformer blocks
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        # Final layer normalization
        self.final_norm = LayerNorm(cfg["emb_dim"])

        # Output projection layer
        # Maps embeddings back to vocabulary logits
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):

        # Input shape: (batch_size, sequence_length)
        batch_size, seq_len = in_idx.shape

        # Convert token IDs into embeddings
        # Shape: (batch, seq_len, emb_dim)
        tok_embeds = self.tok_emb(in_idx)

        # Generate position indices (0...seq_len-1)
        # Then convert them to positional embeddings
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )

        # Combine token and positional embeddings
        x = tok_embeds + pos_embeds

        # Apply dropout
        x = self.drop_emb(x)

        # Pass through transformer blocks
        x = self.trf_blocks(x)

        # Final normalization
        x = self.final_norm(x)

        # Project embeddings to vocabulary logits
        logits = self.out_head(x)

        return logits

## From Chapter 5

In [9]:
# Generates text autoregressively using the trained model
def generate(model, idx, max_new_tokens, context_size,
     temperature=0.0, top_k=None, eos_id=None):

    # Generate tokens one-by-one up to max_new_tokens
    for _ in range(max_new_tokens):

        # Keep only the last context_size tokens (model context window)
        idx_cond = idx[:, -context_size:]

        # Disable gradient tracking for faster inference
        with torch.no_grad():
            logits = model(idx_cond)

        # Take logits for the last generated position
        logits = logits[:, -1, :]

        # Top-k filtering: keep only the k highest probability tokens
        if top_k is not None:
            top_logits, _ = torch.topk(logits, top_k)
            min_val = top_logits[:, -1]

            # Set logits outside top-k to -inf so they cannot be sampled
            logits = torch.where(
                logits < min_val,
                torch.tensor(float('-inf')).to(logits.device),
                logits
            )

        # Temperature sampling (adds randomness to generation)
        if temperature > 0.0:
            logits = logits / temperature

            # Convert logits to probabilities
            probs = torch.softmax(logits, dim=-1)

            # Sample next token from probability distribution
            idx_next = torch.multinomial(probs, num_samples=1)

        else:
            # Greedy decoding: pick the token with highest probability
            idx_next = torch.argmax(logits, dim=-1, keepdim=True)

        # Stop generation if end-of-sequence token is produced
        if idx_next == eos_id:
            break

        # Append generated token to the sequence
        idx = torch.cat((idx, idx_next), dim=1)

    return idx

# Converts input text into token IDs tensor
def text_to_token_ids(text, tokenizer):

    # Encode text using tokenizer (allowing special tokens)
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})

    # Convert to tensor and add batch dimension
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)

    return encoded_tensor


# Converts token IDs back into readable text
def token_ids_to_text(token_ids, tokenizer):

    # Remove batch dimension
    flat = token_ids.squeeze(0)

    # Decode tokens into string
    return tokenizer.decode(flat.tolist())

## Using the model

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = tiktoken.get_encoding("gpt2")
print("cuda" if torch.cuda.is_available() else "cpu")

cuda


In [11]:
model = GPTModel(BASE_CONFIG)

In [12]:
#load model
model_state_dict = torch.load(model_file_path, map_location=device)
model.load_state_dict(model_state_dict)
model.to(device)
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 1024)
  (pos_emb): Embedding(1024, 1024)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(in_features=1024, out_features=1024, bias=True)
        (W_key): Linear(in_features=1024, out_features=1024, bias=True)
        (W_value): Linear(in_features=1024, out_features=1024, bias=True)
        (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=1024, out_features=4096, bias=True)
          (1): GELU()
          (2): Linear(in_features=4096, out_features=1024, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (W_query): Linear(i

In [47]:
def ask_model(instruction, input_text, model, tokenizer, device, temperature):
    prompt = (
        f"Below is an instruction that describes a task."
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{instruction}"
        f"\n\n### Input:\n{input_text}"
        f"\n\n### Response:\n"
    )

    # Convert to tokens
    encoded_input = text_to_token_ids(prompt, tokenizer).to(device)
    context_size = BASE_CONFIG["context_length"]

    # Generate tokens
    with torch.no_grad():
        generated_ids = generate(
            model=model,
            idx=encoded_input,
            max_new_tokens=50,  # Limits how long the response can be
            context_size=context_size,
            temperature=temperature, 
            top_k=None,
            eos_id=50256        # End of text token
        )

    # Convert back to readable English
    decoded_text = token_ids_to_text(generated_ids, tokenizer)

    # Slice off the prompt so we only see the model's new text
    response_only = decoded_text[len(prompt):].strip()

    # Failsafe: Sometimes models babble after answering. Cut it off at the first newline.
    clean_response = response_only.split("<|endoftext|>")[0].split("\n")[0].strip()

    return clean_response

## Testing Grammar Correction

In [48]:
grammar_instruction = "Correct the grammar of the text."

In [54]:
manual_test_sentences_grammar = [
    "He don't know nothing about math.",
    "I am used to wake up early.",
    "If I would have more time I will probably learn another language.",
    "The more you practice, the better you will become it.",
    "Me and my friend we went to cafe and we was talking about different things for hours",
]

for sentence in manual_test_sentences_grammar:
    print(f"Original:   {sentence}")
    print(f"Correction: {ask_model(grammar_instruction, sentence, model, tokenizer, device, 0.0)}")
    print("-" * 50)

Original:   He don't know nothing about math.
Correction: He doesn't know anything about math.
--------------------------------------------------
Original:   I am used to wake up early.
Correction: I am used to waking up early.
--------------------------------------------------
Original:   If I would have more time I will probably learn another language.
Correction: If I had more time I would probably learn another language.
--------------------------------------------------
Original:   The more you practice, the better you will become it.
Correction: The more you practice, the better you will become.
--------------------------------------------------
Original:   Me and my friend we went to cafe and we was talking about different things for hours
Correction: Me and my friend we went to cafe and we were talking about different things for hours.
--------------------------------------------------


In [46]:
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.gleu_score import sentence_gleu

# 1. Define your test cases with the "Gold Standard" answers
grammar_test_cases = [
    {
        "original": "He don't know nothing about math.",
        "reference": "He doesn't know anything about math."
    },
    {
        "original": "I is going to the stores tomorrow.",
        "reference": "I am going to the store tomorrow."
    },
    {
        "original": "She play tennis very good.",
        "reference": "She plays tennis very well."
    },
    {
        "original": "They was late to the meeting.",
        "reference": "They were late to the meeting."
    },
    {
        "original": "I have did my homework already.",
        "reference": "I have done my homework already."
    },
    {
        "original": "He go to school every days.",
        "reference": "He goes to school every day."
    },
    {
        "original": "There is many reasons to learn English.",
        "reference": "There are many reasons to learn English."
    },
    {
        "original": "She don't likes coffee.",
        "reference": "She doesn't like coffee."
    },
    {
        "original": "We was watching a movie when he arrive.",
        "reference": "We were watching a movie when he arrived."
    },
    {
        "original": "I am agree with you.",
        "reference": "I agree with you."
    },
    {
        "original": "He has less friends than me.",
        "reference": "He has fewer friends than I do."
    },
    {
        "original": "She can to speak three languages.",
        "reference": "She can speak three languages."
    },
    {
        "original": "I look forward to see you.",
        "reference": "I look forward to seeing you."
    },
    {
        "original": "This informations are important.",
        "reference": "This information is important."
    },
    {
        "original": "He explained me the problem.",
        "reference": "He explained the problem to me."
    },
    {
        "original": "I didn't went to the party.",
        "reference": "I didn't go to the party."
    },
    {
        "original": "She has 20 years old.",
        "reference": "She is 20 years old."
    },
    {
        "original": "We discussed about the issue.",
        "reference": "We discussed the issue."
    },
    {
        "original": "He is married with a doctor.",
        "reference": "He is married to a doctor."
    },
    {
        "original": "I have much friends in this city.",
        "reference": "I have many friends in this city."
    },
    {
        "original": "She suggested me to go home.",
        "reference": "She suggested that I go home."
    },
    {
        "original": "He is more taller than his brother.",
        "reference": "He is taller than his brother."
    },
    {
        "original": "I enjoy to play football.",
        "reference": "I enjoy playing football."
    },
    {
        "original": "The news are very shocking.",
        "reference": "The news is very shocking."
    },
    {
        "original": "He didn't knew the answer.",
        "reference": "He didn't know the answer."
    },
    {
        "original": "I have been knowing him for five years.",
        "reference": "I have known him for five years."
    },
    {
        "original": "She insisted to pay for dinner.",
        "reference": "She insisted on paying for dinner."
    },
    {
        "original": "This is the first time I visit this city.",
        "reference": "This is the first time I have visited this city."
    },
    {
        "original": "He suggested me to apply for the job.",
        "reference": "He suggested that I apply for the job."
    },
    {
        "original": "I am used to wake up early.",
        "reference": "I am used to waking up early."
    },
    {
        "original": "The amount of people here is overwhelming.",
        "reference": "The number of people here is overwhelming."
    },
    {
        "original": "She has a strong accent but it's easy understandable.",
        "reference": "She has a strong accent but it's easily understandable."
    },
    {
        "original": "He is responsible of managing the team.",
        "reference": "He is responsible for managing the team."
    },
    {
        "original": "I would rather to stay at home tonight.",
        "reference": "I would rather stay at home tonight."
    },
    {
        "original": "Despite of the rain, we went out.",
        "reference": "Despite the rain, we went out."
    },
    {
        "original": "She explained me why she was late.",
        "reference": "She explained to me why she was late."
    },
    {
        "original": "He denied to break the window.",
        "reference": "He denied breaking the window."
    },
    {
        "original": "I am looking forward to meet you.",
        "reference": "I am looking forward to meeting you."
    },
    {
        "original": "The project was more difficult than we expected it would be.",
        "reference": "The project was more difficult than we expected."
    },
    {
        "original": "He didn't manage finishing the task on time.",
        "reference": "He didn't manage to finish the task on time."
    },
    {
        "original": "She is one of the best student in the class.",
        "reference": "She is one of the best students in the class."
    },
    {
        "original": "Hardly I had arrived when it started to rain.",
        "reference": "Hardly had I arrived when it started to rain."
    },
    {
        "original": "No sooner she left than he called.",
        "reference": "No sooner had she left than he called."
    },
    {
        "original": "It depends from the situation.",
        "reference": "It depends on the situation."
    },
    {
        "original": "He is used to work under pressure.",
        "reference": "He is used to working under pressure."
    },
    {
        "original": "I regret to tell him the truth yesterday.",
        "reference": "I regret telling him the truth yesterday."
    },
    {
        "original": "She made me to apologize.",
        "reference": "She made me apologize."
    },
    {
        "original": "The more you practice, the better you will become it.",
        "reference": "The more you practice, the better you will become."
    },
    {
        "original": "He has difficulty to understand complex texts.",
        "reference": "He has difficulty understanding complex texts."
    },
    {
        "original": "I wish I would know the answer.",
        "reference": "I wish I knew the answer."
    }
]

In [26]:
gleu_scores = []
bleu_scores = []
for case in grammar_test_cases:
    # Get the model's output
    prediction = ask_model(grammar_instruction, case["original"], model, tokenizer, device)
    
    # Prepare tokens for NLTK (it expects lists of words)
    # References must be a list of lists: [[word, word]]
    ref_tokens = [case["reference"].split()]
    pred_tokens = prediction.split()
    
    # Calculate scores (0.0 to 1.0)
    # Note: we use 'sentence_' versions here instead of 'corpus_'
    score_gleu = sentence_gleu(ref_tokens, pred_tokens) * 100
    gleu_scores.append(score_gleu)
    score_bleu = sentence_bleu(ref_tokens, pred_tokens) * 100
    bleu_scores.append(score_bleu)
    

# Calculate averages using standard Python
avg_gleu = sum(gleu_scores) / len(gleu_scores)
avg_bleu = sum(bleu_scores) / len(bleu_scores)

print("-" * 50)
print(f"AVERAGE SCORES (Manual Tests)")
print(f"GLEU: {avg_gleu:.2f} | BLEU: {avg_bleu:.2f}")
print("-" * 50)

--------------------------------------------------
AVERAGE SCORES (Manual Tests)
GLEU: 72.56 | BLEU: 60.88
--------------------------------------------------


## Testing Naturalness Instruction

In [56]:
naturalness_instruction = "Rewrite the text to sound more natural and fluent while preserving meaning."

In [116]:
manual_test_sentences_naturalness = [
    "As I walked to the store, I noticed an unmistakable puff of chill frost crust covering the surface of the door.",
    "Finally, the children started singing. The children danced vividly too",
]

for sentence in manual_test_sentences_naturalness:
    print(f"Original:   {sentence}")
    print(f"Correction: {ask_model(naturalness_instruction, sentence, model, tokenizer, device, 0.4)}")
    print("-" * 50)

Original:   As I walked to the store, I noticed an unmistakable puff of chill frost crust covering the surface of the door.
Correction: I walked to the store and noticed an unmistakable puff of chill frost crust covering the surface of the door.
--------------------------------------------------
Original:   Finally, the children started singing. The children danced vividly too
Correction: The children started singing and dancing.
--------------------------------------------------


## Testing Academic Instruction

In [27]:
academic_instruction = "Rewrite the text in polished academic English."

In [104]:
manual_test_sentences_academic = [
    ["The experiment result show significant improvements.", 0.4],
    ["This paper try to analyze how social media affect students performance.", 0.0],
    ["We did a bunch of tests and the numbers show that our approach is the best so far.", 0.7]
]

for sentence, temp in manual_test_sentences_academic:
    print(f"Original:   {sentence}")
    print(f"Correction: {ask_model(academic_instruction, sentence, model, tokenizer, device, temp)}")
    print("-" * 50)

Original:   The experiment result show significant improvements.
Correction: The results show significant improvements in the accuracy of the subjects' responses.
--------------------------------------------------
Original:   This paper try to analyze how social media affect students performance.
Correction: This paper analyze how social media affect students performance.
--------------------------------------------------
Original:   We did a bunch of tests and the numbers show that our approach is the best so far.
Correction: We performed a bunch of tests and the results showed that the approach we used is the best so far.
--------------------------------------------------
